In [23]:
# scikit-learn
# pytorch
# pandas
# numpy
from sklearn.model_selection import train_test_split
import torch          # basic pytorch functionality
import torch.nn as nn # neural networks
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd   # dataframe manipulation
import torch.optim as optim

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.LSTM(),
            nn.ReLU,
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, x_data):
        results = self.model(x_data)
        return results

In [ ]:
import yfinance as yf
data = yf.download(tickers='^GSPC', start='2005-01-01', end='2025-01-01')['Close']
x_data = data[:-1].pct_change()     # n-1
y_data = data.shift(1).dropna().pct_change() # n
x_data.dropna(inplace=True)
y_data.dropna(inplace=True)
# Standard Scaler --> normalizes values
# PCA --> reduces redundancy

C:\Users\hudso\AppData\Local\Temp\ipykernel_25668\376510746.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers='^GSPC', start='2005-01-01', end='2025-01-01')['Close']
[*********************100%***********************]  1 of 1 completed


In [21]:
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32).view(-1, 1)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [22]:
input_dim = X_train_tensor.shape[1]
print(x_data.head())

Ticker         ^GSPC
Date                
2005-01-04 -0.011671
2005-01-05 -0.003628
2005-01-06  0.003506
2005-01-07 -0.001431
2005-01-10  0.003423


In [34]:
input_dim = X_train_tensor.shape[1]
output_dim = 1
model = SimpleNN(input_dim, output_dim)
optimizer = optim.Adam(params=model.parameters(), lr=1e-3)
loss_func = nn.MSELoss()

In [ ]:
EPOCHS = 100
for epoch in range(EPOCHS):
    epoch_loss = 0
    optimizer.zero_grad
    for batch_x, batch_y in train_loader:
        predictions = model(batch_x)
        batch_loss = loss_func(predictions, batch_y)
        epoch_loss += batch_loss

    if (epoch % 10 == 0):
        print(f'{epoch}: {epoch_loss}')


0: 2.060654878616333
10: 2.0603713989257812
20: 2.0603601932525635
30: 2.060744524002075
40: 2.061317205429077
50: 2.060687780380249
60: 2.0605828762054443
70: 2.0608060359954834
80: 2.0604515075683594
90: 2.0603835582733154


In [38]:
# Test
accuracy = 0
for batch_x, batch_y in test_loader:
    predictions = model(batch_x)
    batch_loss = loss_func(predictions, batch_y)
    accuracy += batch_loss
print(accuracy)

tensor(0.5255, grad_fn=<AddBackward0>)
